# Tool-Use GRPO Training with Qwen3-8BTrain a larger language model to generate correct JSON tool calls using GRPO.- **Model**: Qwen/Qwen3-8B (4-bit quantized for A100 80GB)- **Task**: JSON tool-call generation across 15 tool types- **Method**: TRL GRPOTrainer with LoRA (rank 32) + BitsAndBytes 4-bit- **Logging**: Weights & Biases (project: tinker-rl-agentic)- **Hub**: arvindcr4/

In [ ]:
# Cell 1: Install dependencies!pip install -q trl transformers peft datasets accelerate bitsandbytes wandb huggingface_hub

In [ ]:
# Cell 2: GPU checkimport torchprint(f"CUDA available: {torch.cuda.is_available()}")print(f"GPU: {torch.cuda.get_device_name(0)}")print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Cell 3: Auto-login with pre-configured keys
import os

# Set your API keys here or use Colab secrets
# In Colab: Runtime > Manage secrets, add WANDB_API_KEY and HF_TOKEN
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except:
    # Fallback: set manually
    os.environ.setdefault('WANDB_API_KEY', 'YOUR_WANDB_API_KEY')
    os.environ.setdefault('HF_TOKEN', 'YOUR_HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

In [ ]:
# Cell 4: Define 15 tool types and create 40 training examplesimport jsonimport randomTOOLS = {    "web_search": {"query": "string", "num_results": "integer"},    "send_email": {"to": "string", "subject": "string", "body": "string"},    "create_calendar_event": {"title": "string", "date": "string", "time": "string"},    "get_weather": {"location": "string", "unit": "string"},    "translate_text": {"text": "string", "source_lang": "string", "target_lang": "string"},    "calculate": {"expression": "string"},    "set_reminder": {"message": "string", "datetime": "string"},    "read_file": {"file_path": "string"},    "write_file": {"file_path": "string", "content": "string"},    "list_directory": {"path": "string", "recursive": "boolean"},    "http_request": {"url": "string", "method": "string", "headers": "object"},    "database_query": {"query": "string", "database": "string"},    "image_generate": {"prompt": "string", "width": "integer", "height": "integer"},    "code_execute": {"language": "string", "code": "string"},    "summarize_text": {"text": "string", "max_length": "integer"}}TOOL_DESCRIPTIONS = {    "web_search": "Search the web for information",    "send_email": "Send an email to a recipient",    "create_calendar_event": "Create a calendar event",    "get_weather": "Get weather information for a location",    "translate_text": "Translate text between languages",    "calculate": "Evaluate a mathematical expression",    "set_reminder": "Set a reminder for a specific time",    "read_file": "Read contents of a file",    "write_file": "Write content to a file",    "list_directory": "List files in a directory",    "http_request": "Make an HTTP request to a URL",    "database_query": "Execute a database query",    "image_generate": "Generate an image from a text prompt",    "code_execute": "Execute code in a specified language",    "summarize_text": "Summarize a piece of text"}# 40 training examples across all 15 tool typesTRAINING_DATA = [    {"prompt": "Search for the latest news about AI safety", "tool": "web_search", "args": {"query": "latest news AI safety", "num_results": 5}},    {"prompt": "Find information about Python 3.12 new features", "tool": "web_search", "args": {"query": "Python 3.12 new features", "num_results": 10}},    {"prompt": "Search for transformer architecture papers", "tool": "web_search", "args": {"query": "transformer architecture papers", "num_results": 5}},    {"prompt": "Send an email to john@example.com about the meeting tomorrow", "tool": "send_email", "args": {"to": "john@example.com", "subject": "Meeting Tomorrow", "body": "Hi John, just a reminder about our meeting tomorrow. Looking forward to it."}},    {"prompt": "Email sarah@company.com the quarterly report summary", "tool": "send_email", "args": {"to": "sarah@company.com", "subject": "Quarterly Report Summary", "body": "Hi Sarah, please find the quarterly report summary attached."}},    {"prompt": "Create a meeting for March 15th at 2pm called Team Standup", "tool": "create_calendar_event", "args": {"title": "Team Standup", "date": "2025-03-15", "time": "14:00"}},    {"prompt": "Schedule a dentist appointment on April 1st at 10am", "tool": "create_calendar_event", "args": {"title": "Dentist Appointment", "date": "2025-04-01", "time": "10:00"}},    {"prompt": "Add a birthday party event on December 25th at 6pm", "tool": "create_calendar_event", "args": {"title": "Birthday Party", "date": "2025-12-25", "time": "18:00"}},    {"prompt": "What is the weather like in San Francisco?", "tool": "get_weather", "args": {"location": "San Francisco", "unit": "fahrenheit"}},    {"prompt": "Check the weather in Tokyo in celsius", "tool": "get_weather", "args": {"location": "Tokyo", "unit": "celsius"}},    {"prompt": "Get weather forecast for London", "tool": "get_weather", "args": {"location": "London", "unit": "celsius"}},    {"prompt": "Translate Hello how are you from English to Spanish", "tool": "translate_text", "args": {"text": "Hello, how are you?", "source_lang": "en", "target_lang": "es"}},    {"prompt": "Translate Good morning to Japanese from English", "tool": "translate_text", "args": {"text": "Good morning", "source_lang": "en", "target_lang": "ja"}},    {"prompt": "Convert Bonjour le monde from French to English", "tool": "translate_text", "args": {"text": "Bonjour le monde", "source_lang": "fr", "target_lang": "en"}},    {"prompt": "Calculate 15 * 32 + 7", "tool": "calculate", "args": {"expression": "15 * 32 + 7"}},    {"prompt": "What is the square root of 144?", "tool": "calculate", "args": {"expression": "sqrt(144)"}},    {"prompt": "Compute 2 to the power of 10", "tool": "calculate", "args": {"expression": "2**10"}},    {"prompt": "Remind me to call mom at 5pm today", "tool": "set_reminder", "args": {"message": "Call mom", "datetime": "2025-03-14T17:00:00"}},    {"prompt": "Set a reminder for the standup meeting at 9am tomorrow", "tool": "set_reminder", "args": {"message": "Standup meeting", "datetime": "2025-03-15T09:00:00"}},    {"prompt": "Read the file at /home/user/config.yaml", "tool": "read_file", "args": {"file_path": "/home/user/config.yaml"}},    {"prompt": "Show me the contents of /etc/hosts", "tool": "read_file", "args": {"file_path": "/etc/hosts"}},    {"prompt": "Write Hello World to /tmp/test.txt", "tool": "write_file", "args": {"file_path": "/tmp/test.txt", "content": "Hello World"}},    {"prompt": "Save the configuration to /home/user/settings.json", "tool": "write_file", "args": {"file_path": "/home/user/settings.json", "content": "{\"theme\": \"dark\", \"language\": \"en\"}"}},    {"prompt": "List all files in /home/user/documents", "tool": "list_directory", "args": {"path": "/home/user/documents", "recursive": false}},    {"prompt": "Show all files recursively in the project directory /app/src", "tool": "list_directory", "args": {"path": "/app/src", "recursive": true}},    {"prompt": "Make a GET request to https://api.example.com/users", "tool": "http_request", "args": {"url": "https://api.example.com/users", "method": "GET", "headers": {}}},    {"prompt": "POST to https://api.example.com/data with auth header", "tool": "http_request", "args": {"url": "https://api.example.com/data", "method": "POST", "headers": {"Authorization": "Bearer token123"}}},    {"prompt": "Fetch data from https://api.github.com/repos", "tool": "http_request", "args": {"url": "https://api.github.com/repos", "method": "GET", "headers": {"Accept": "application/json"}}},    {"prompt": "Run SQL query to select all users from the main database", "tool": "database_query", "args": {"query": "SELECT * FROM users", "database": "main"}},    {"prompt": "Query the analytics database for page views in March", "tool": "database_query", "args": {"query": "SELECT * FROM page_views WHERE month = 'March'", "database": "analytics"}},    {"prompt": "Generate an image of a sunset over mountains, 1024x768", "tool": "image_generate", "args": {"prompt": "a sunset over mountains, photorealistic", "width": 1024, "height": 768}},    {"prompt": "Create a 512x512 image of a cute robot", "tool": "image_generate", "args": {"prompt": "a cute friendly robot, digital art", "width": 512, "height": 512}},    {"prompt": "Run this Python code to print hello", "tool": "code_execute", "args": {"language": "python", "code": "print('hello')"}},    {"prompt": "Execute JavaScript code to log 2+2", "tool": "code_execute", "args": {"language": "javascript", "code": "console.log(2+2)"}},    {"prompt": "Run Python to print pi", "tool": "code_execute", "args": {"language": "python", "code": "import math; print(math.pi)"}},    {"prompt": "Summarize this article about climate change in 100 words", "tool": "summarize_text", "args": {"text": "Climate change is a long-term shift in global temperatures and weather patterns. While natural cycles have always influenced Earth's climate, human activities have been the main driver since the 1800s, primarily due to burning fossil fuels like coal, oil, and gas.", "max_length": 100}},    {"prompt": "Give me a brief summary of quantum computing in 50 words", "tool": "summarize_text", "args": {"text": "Quantum computing harnesses quantum mechanical phenomena such as superposition and entanglement to process information. Unlike classical computers that use bits, quantum computers use qubits which can exist in multiple states simultaneously, potentially solving certain problems exponentially faster.", "max_length": 50}},    {"prompt": "Search online for best practices in prompt engineering", "tool": "web_search", "args": {"query": "best practices prompt engineering LLM", "num_results": 5}},    {"prompt": "List files in /tmp/cache directory", "tool": "list_directory", "args": {"path": "/tmp/cache", "recursive": false}},    {"prompt": "Remind me to submit the report by Friday 3pm", "tool": "set_reminder", "args": {"message": "Submit the report", "datetime": "2025-03-21T15:00:00"}}]TOOLS_PROMPT = "Available tools:\n"for tool_name, params in TOOLS.items():    desc = TOOL_DESCRIPTIONS[tool_name]    params_str = json.dumps(params)    TOOLS_PROMPT += f"- {tool_name}: {desc}. Parameters: {params_str}\n"print(f"Created {len(TRAINING_DATA)} training examples across {len(TOOLS)} tool types")print(f"\nTool types: {list(TOOLS.keys())}")

In [ ]:
# Cell 5: Define reward functionimport jsonimport redef extract_tool_call(text):    """Extract tool name and arguments from model output."""    try:        # Look for ```json ... ``` blocks        json_match = re.search(r'```json\s*\n?(.*?)\n?```', text, re.DOTALL)        if json_match:            parsed = json.loads(json_match.group(1).strip())            return parsed.get("tool") or parsed.get("name"), parsed.get("args") or parsed.get("arguments") or parsed.get("parameters", {})        # Look for raw JSON object        json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text)        if json_match:            parsed = json.loads(json_match.group(0))            return parsed.get("tool") or parsed.get("name"), parsed.get("args") or parsed.get("arguments") or parsed.get("parameters", {})    except (json.JSONDecodeError, AttributeError):        pass    return None, Nonedef tool_use_reward_fn(completions, **kwargs):    """    Reward function for tool-use GRPO.    - 1.0: correct tool + valid JSON    - 0.25: valid JSON but wrong tool    - 0.0: invalid output    """    prompts = kwargs.get("prompts", [])    rewards = []    for i, completion in enumerate(completions):        if isinstance(completion, list):            text = completion[-1]["content"] if completion else ""        else:            text = completion        tool_name, args = extract_tool_call(text)        if tool_name is None:            rewards.append(0.0)            continue        prompt_text = prompts[i] if i < len(prompts) else ""        if isinstance(prompt_text, list):            prompt_text = prompt_text[-1]["content"] if prompt_text else ""        expected_tool = None        for example in TRAINING_DATA:            if example["prompt"] in prompt_text:                expected_tool = example["tool"]                break        if expected_tool and tool_name == expected_tool:            rewards.append(1.0)        elif tool_name in TOOLS:            rewards.append(0.25)        else:            rewards.append(0.0)    return rewardsprint("Reward function defined.")print("  1.0  = correct tool + valid JSON")print("  0.25 = valid JSON but wrong tool")print("  0.0  = invalid output")

In [ ]:
# Cell 6: Format dataset for GRPOTrainerfrom datasets import Datasetdef format_prompt(example):    """Format a training example into a chat-style prompt."""    system_msg = (        "You are a helpful assistant that can use tools. "        "When the user asks you to do something, respond with a JSON tool call.\n"        "Format your response as:\n"        '```json\n{"tool": "<tool_name>", "args": {<arguments>}}\n```\n\n'        f"{TOOLS_PROMPT}"    )    return [        {"role": "system", "content": system_msg},        {"role": "user", "content": example["prompt"]}    ]formatted_data = []for example in TRAINING_DATA:    formatted_data.append({        "prompt": format_prompt(example),        "expected_tool": example["tool"],        "expected_args": json.dumps(example["args"])    })dataset = Dataset.from_list(formatted_data)print(f"Dataset size: {len(dataset)}")print(f"Sample prompt: {dataset[0]['prompt'][-1]['content']}")

In [ ]:
# Cell 7: Load model and tokenizer with 4-bit quantizationfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfigMODEL_NAME = "Qwen/Qwen3-8B"# 4-bit quantization config for A100 80GBbnb_config = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_quant_type="nf4",    bnb_4bit_compute_dtype=torch.bfloat16,    bnb_4bit_use_double_quant=True)tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenmodel = AutoModelForCausalLM.from_pretrained(    MODEL_NAME,    quantization_config=bnb_config,    device_map="auto",    trust_remote_code=True)print(f"Model loaded: {MODEL_NAME} (4-bit quantized)")print(f"Parameters: {model.num_parameters():,}")

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training# Prepare model for k-bit trainingmodel = prepare_model_for_kbit_training(model)# Cell 8: LoRA configurationlora_config = LoraConfig(    r=32,    lora_alpha=64,    lora_dropout=0.05,    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],    bias="none",    task_type="CAUSAL_LM")print(f"LoRA config: rank={lora_config.r}, alpha={lora_config.lora_alpha}")print(f"Target modules: {lora_config.target_modules}")

In [ ]:
# Cell 9: GRPO configurationfrom trl import GRPOConfiggrpo_config = GRPOConfig(    output_dir="./grpo-tool-use-qwen3-8b",    num_train_epochs=3,    per_device_train_batch_size=2,    gradient_accumulation_steps=4,    learning_rate=4e-5,    lr_scheduler_type="cosine",    warmup_ratio=0.1,    max_completion_length=256,    num_generations=16,    logging_steps=1,    save_steps=50,    bf16=True,    report_to="wandb",    run_name="tool-use-grpo-qwen3-8b",    seed=42,    remove_unused_columns=False,    gradient_checkpointing=True)print(f"GRPO Config:")print(f"  Epochs: {grpo_config.num_train_epochs}")print(f"  Batch size: {grpo_config.per_device_train_batch_size}")print(f"  LR: {grpo_config.learning_rate}")print(f"  Group size: {grpo_config.num_generations}")

In [ ]:
# Cell 10: Create GRPOTrainer and trainfrom trl import GRPOTraineros.environ["WANDB_PROJECT"] = "tinker-rl-agentic"trainer = GRPOTrainer(    model=model,    args=grpo_config,    train_dataset=dataset,    reward_funcs=tool_use_reward_fn,    peft_config=lora_config,    processing_class=tokenizer)print("Starting GRPO training...")trainer.train()print("Training complete!")

In [ ]:
# Cell 11: Save the trained modelOUTPUT_DIR = "./grpo-tool-use-qwen3-8b-final"trainer.save_model(OUTPUT_DIR)tokenizer.save_pretrained(OUTPUT_DIR)print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# Cell 12: Evaluation - compare base vs GRPO-trained modelfrom transformers import pipelinetrained_pipe = pipeline(    "text-generation",    model=OUTPUT_DIR,    tokenizer=tokenizer,    device_map="auto",    torch_dtype=torch.bfloat16,    model_kwargs={"quantization_config": bnb_config})test_prompts = [    "Search for recipes for chocolate cake",    "Send an email to boss@company.com about project status",    "What is the weather in New York?",    "Translate Thank you to German",    "Calculate 256 / 8 + 3"]print("=" * 80)print("EVALUATION: GRPO-trained Qwen3-8B")print("=" * 80)base_correct = 0trained_correct = 0for prompt_text in test_prompts:    messages = format_prompt({"prompt": prompt_text})    trained_out = trained_pipe(messages, max_new_tokens=256, do_sample=True, temperature=0.7)    trained_text = trained_out[0]["generated_text"][-1]["content"] if isinstance(trained_out[0]["generated_text"], list) else trained_out[0]["generated_text"]    trained_tool, _ = extract_tool_call(trained_text)    if trained_tool and trained_tool in TOOLS:        trained_correct += 1    print(f"\nPrompt: {prompt_text}")    print(f"  Trained model tool: {trained_tool}")print(f"\n{'=' * 80}")print(f"Trained accuracy: {trained_correct}/{len(test_prompts)} ({100*trained_correct/len(test_prompts):.1f}%)")

In [ ]:
# Cell 13: Push to HuggingFace HubHUB_REPO = "arvindcr4/tool-use-grpo-qwen3-8b"trainer.push_to_hub(HUB_REPO)tokenizer.push_to_hub(HUB_REPO)print(f"Model pushed to: https://huggingface.co/{HUB_REPO}")wandb.finish()print("Done! W&B run finished.")